# 9.7 前缀缓存 (Prefix Caching / RadixAttention)

> 🕐 预估学习时间：30分钟

前缀缓存是推理优化中性价比极高的技术：当多个请求共享相同的前缀（如系统提示、few-shot 示例）时，可以复用已计算的 KV 缓存，避免重复前向计算。

本节涵盖：
- 前缀缓存原理与动机
- KV Cache 复用机制
- RadixAttention（基数树管理前缀）
- 缓存淘汰策略（LRU / LFU / 自适应）
- 多轮对话场景的缓存优化

代表系统：SGLang、vLLM（Automatic Prefix Caching）、TensorRT-LLM。

## 1. 前缀缓存概述

**为什么需要前缀缓存？**

实际 LLM 服务中，大量请求共享相同前缀：
- **系统提示**：所有请求都使用相同的 system prompt（几百到几千 token）
- **Few-shot 示例**：相同的示例序列被反复使用
- **多轮对话**：历史对话内容是新一轮请求的前缀
- **文档问答**：同一文档被多次提问

**核心思想**：将这些共享前缀对应的 KV 缓存计算一次并保存，后续请求直接复用，跳过前缀部分的前向计算。

**收益**：
- 首 token 延迟（TTFT）大幅降低
- 吞吐量显著提升（前缀部分计算被摊销）
- 显存占用减少（避免重复存储相同 KV）

**典型场景加速比**：
- 共享 1K token 系统提示：2-5x 加速
- 共享 4K token 文档：5-10x 加速
- 多轮对话（上下文增长）：每轮累积加速

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

torch.manual_seed(42)


class SimpleAttention(nn.Module):
    '''简化版多头注意力，用于演示前缀缓存。'''
    def __init__(self, d_model=128, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.out_proj(out)


class NaiveInferenceEngine:
    '''基线：每次请求都从头计算 KV 缓存。'''
    def __init__(self, d_model=128, n_heads=4):
        self.attn = SimpleAttention(d_model, n_heads)
        self.embed = nn.Embedding(1000, d_model)
        self.compute_count = 0

    def compute_kv(self, token_ids):
        '''对完整序列计算 KV，返回耗时。'''
        x = self.embed(token_ids.unsqueeze(0))
        start = time.perf_counter()
        _ = self.attn(x)
        self.compute_count += token_ids.shape[0]
        return time.perf_counter() - start

    def serve(self, requests):
        total_time = 0.0
        for token_ids in requests:
            total_time += self.compute_kv(token_ids)
        return total_time


# 模拟场景：共享系统提示 + 不同用户查询
system_prompt = torch.randint(0, 1000, (256,))
user_queries = [torch.randint(0, 1000, (32,)) for _ in range(8)]
requests = [torch.cat([system_prompt, q]) for q in user_queries]

engine = NaiveInferenceEngine()
naive_time = engine.serve(requests)

print('=== 基线：无前缀缓存 ===')
print(f'系统提示长度: {system_prompt.shape[0]} tokens')
print(f'用户查询数: {len(user_queries)}, 每个长度: {user_queries[0].shape[0]} tokens')
print(f'总计算 token 数: {engine.compute_count}')
print(f'总耗时: {naive_time*1000:.2f} ms')
print(f'\nKey: 基线每次请求都重新计算整个序列，系统提示被重复计算 {len(user_queries)} 次。')
print(f'前缀缓存的目标就是消除这种重复计算。')

## 2. KV Cache 复用原理

**前缀匹配**：给定一个新请求的 token 序列，找到缓存中最长的匹配前缀，复用其 KV，仅对剩余部分计算新的 KV。

**缓存键（Cache Key）**：
- 通常使用 token 序列本身的哈希（或逐 token 的滚动哈希）
- 不同请求只要前缀 token 完全一致即可共享 KV
- 注意：相同 token 序列 + 相同模型权重 → 相同 KV

**缓存失效**：
- 模型权重更新（如微调后）→ 全部失效
- 量化精度改变 → 全部失效
- 注意力实现不同（如 GQA 分组变化）→ 全部失效

**匹配流程**：
1. 对输入 token 序列逐 token 计算哈希
2. 在缓存中查找最长前缀匹配
3. 复用匹配部分的 KV，仅计算新增部分
4. 将新计算的 KV 加入缓存

**与 PagedAttention 的关系**：PagedAttention 提供了块级显存管理，使前缀缓存可以以页为单位共享，避免大块拷贝。

In [ ]:
import torch
import torch.nn as nn
import math
import time
import hashlib

torch.manual_seed(42)


class PrefixCacheEngine:
    '''基础前缀缓存引擎：按 token 前缀缓存 KV。'''
    def __init__(self, d_model=128, n_heads=4):
        self.attn = SimpleAttention(d_model, n_heads)
        self.embed = nn.Embedding(1000, d_model)
        self.kv_cache = {}
        self.compute_count = 0
        self.cache_hits = 0
        self.cache_misses = 0
        self._step = 8

    def _hash(self, token_ids):
        return hashlib.md5(token_ids.numpy().tobytes()).hexdigest()

    def _find_longest_match(self, token_ids):
        '''从长到短查找最长匹配前缀。'''
        n = token_ids.shape[0]
        for length in range(n, 0, -self._step):
            h = self._hash(token_ids[:length])
            if h in self.kv_cache:
                return length
        return 0

    def _cache_prefixes(self, token_ids):
        '''缓存序列的多个前缀哈希，便于后续匹配。'''
        n = token_ids.shape[0]
        for length in range(self._step, n + 1, self._step):
            self.kv_cache[self._hash(token_ids[:length])] = True

    def compute_with_cache(self, token_ids):
        '''带前缀缓存的 KV 计算。'''
        n = token_ids.shape[0]
        match_len = self._find_longest_match(token_ids)

        if match_len >= n:
            self.cache_hits += 1
            return 0.0

        new_part = token_ids[match_len:]
        x = self.embed(new_part.unsqueeze(0))
        start = time.perf_counter()
        _ = self.attn(x)
        elapsed = time.perf_counter() - start
        self.compute_count += new_part.shape[0]

        self._cache_prefixes(token_ids)
        if match_len > 0:
            self.cache_hits += 1
        else:
            self.cache_misses += 1
        return elapsed

    def serve(self, requests):
        total_time = 0.0
        for token_ids in requests:
            total_time += self.compute_with_cache(token_ids)
        return total_time


# 同样的场景：共享系统提示 + 不同用户查询
system_prompt = torch.randint(0, 1000, (256,))
user_queries = [torch.randint(0, 1000, (32,)) for _ in range(8)]
requests = [torch.cat([system_prompt, q]) for q in user_queries]

naive_engine = NaiveInferenceEngine()
naive_time = naive_engine.serve(requests)

cache_engine = PrefixCacheEngine()
cache_time = cache_engine.serve(requests)

print('=== 前缀缓存 vs 基线 ===')
print(f'系统提示: {system_prompt.shape[0]} tokens (共享)')
print(f'用户查询: {len(user_queries)} 个，每个 {user_queries[0].shape[0]} tokens')
print(f'\n基线计算 token 数: {naive_engine.compute_count}')
print(f'缓存引擎计算 token 数: {cache_engine.compute_count}')
print(f'缓存命中: {cache_engine.cache_hits}, 未命中: {cache_engine.cache_misses}')
print(f'\n基线耗时: {naive_time*1000:.2f} ms')
print(f'缓存耗时: {cache_time*1000:.2f} ms')
if cache_time > 0:
    print(f'加速比: {naive_time/cache_time:.2f}x')
print(f'\nKey: 前缀缓存将重复的系统提示计算从 {len(user_queries)} 次降为 1 次。')
print(f'计算量从 {naive_engine.compute_count} 降至 {cache_engine.compute_count} tokens。')

## 3. RadixAttention

**SGLang 的核心创新**：使用基数树（Radix Tree）管理前缀缓存，实现自动前缀去重与共享。

**为什么用基数树？**

简单哈希缓存的问题：
- 只能匹配完整的已缓存前缀
- 不同请求的部分重叠前缀无法自动发现
- 缓存粒度固定，缺乏灵活性

基数树的优势：
- **自动发现任意长度的共享前缀**：树结构天然支持前缀匹配
- **增量更新**：新请求只需遍历树找到分叉点
- **去重**：相同前缀在树中只存一份 KV
- **高效淘汰**：叶子节点可独立淘汰

**基数树结构**：
- 每个节点存储一段 token 序列及其 KV 缓存
- 从根到某节点的路径拼接 = 完整前缀
- 子节点共享父节点的前缀

**工作流程**：
1. 新请求 token 序列从根节点开始匹配
2. 沿树向下，找到最长匹配路径
3. 在分叉点分裂节点，插入新分支
4. 复用匹配部分的 KV，仅计算新增部分

**应用**：SGLang、vLLM 的 Automatic Prefix Caching (APC) 均采用类似思想。

In [ ]:
import torch
import torch.nn as nn
import math
import time

torch.manual_seed(42)


class RadixTreeNode:
    '''基数树节点：存储一段 token 序列。'''
    def __init__(self, token_ids=None):
        self.token_ids = token_ids if token_ids is not None else torch.tensor([], dtype=torch.long)
        self.children = {}
        self.kv_computed = False


class RadixAttentionCache:
    '''基于基数树的前缀缓存管理器（SGLang RadixAttention 简化版）。'''
    def __init__(self, d_model=128, n_heads=4):
        self.attn = SimpleAttention(d_model, n_heads)
        self.embed = nn.Embedding(1000, d_model)
        self.root = RadixTreeNode()
        self.compute_count = 0
        self.reused_count = 0
        self.node_count = 1

    def _split_node(self, node, split_pos):
        '''在 split_pos 处将节点分裂为前缀节点和后缀节点。'''
        old_tokens = node.token_ids
        prefix_node = RadixTreeNode(old_tokens[:split_pos].clone())
        prefix_node.kv_computed = node.kv_computed
        node.token_ids = old_tokens[split_pos:].clone()
        prefix_node.children = {node.token_ids[0].item(): node}
        self.node_count += 1
        return prefix_node

    def insert_and_compute(self, token_ids):
        '''插入序列，复用已缓存前缀，计算新增部分。'''
        n = token_ids.shape[0]
        node = self.root
        pos = 0
        elapsed = 0.0

        while pos < n:
            first_token = token_ids[pos].item()
            if first_token not in node.children:
                new_node = RadixTreeNode(token_ids[pos:].clone())
                new_node.kv_computed = True
                node.children[first_token] = new_node
                self.node_count += 1
                x = self.embed(token_ids[pos:].unsqueeze(0))
                start = time.perf_counter()
                _ = self.attn(x)
                elapsed += time.perf_counter() - start
                self.compute_count += (n - pos)
                return elapsed

            child = node.children[first_token]
            child_tokens = child.token_ids
            remaining_child = child_tokens.shape[0]
            remaining_seq = n - pos
            min_len = min(remaining_child, remaining_seq)

            overlap = 0
            for i in range(min_len):
                if child_tokens[i].item() == token_ids[pos + i].item():
                    overlap += 1
                else:
                    break

            if overlap < remaining_child:
                child = self._split_node(child, overlap)
                node.children[first_token] = child

            self.reused_count += overlap
            pos += overlap
            node = child

        return elapsed

    def serve(self, requests):
        total_time = 0.0
        for token_ids in requests:
            total_time += self.insert_and_compute(token_ids)
        return total_time


# 场景：多种共享前缀模式
system_prompt = torch.randint(0, 1000, (128,))
few_shot = torch.randint(0, 1000, (64,))

reqs_group1 = [torch.cat([system_prompt, few_shot, torch.randint(0, 1000, (16,))]) for _ in range(3)]
reqs_group2 = [torch.cat([system_prompt, torch.randint(0, 1000, (24,))]) for _ in range(2)]
reqs_group3 = [torch.randint(0, 1000, (40,))]
all_requests = reqs_group1 + reqs_group2 + reqs_group3

naive_engine = NaiveInferenceEngine()
naive_time = naive_engine.serve(all_requests)

radix_engine = RadixAttentionCache()
radix_time = radix_engine.serve(all_requests)

print('=== RadixAttention 前缀管理 ===')
print(f'请求总数: {len(all_requests)}')
print(f'  共享 system+few-shot: {len(reqs_group1)} 个')
print(f'  仅共享 system: {len(reqs_group2)} 个')
print(f'  独立请求: {len(reqs_group3)} 个')
print(f'\n基数树节点数: {radix_engine.node_count}')
print(f'复用 token 数: {radix_engine.reused_count}')
print(f'\n基线计算 token: {naive_engine.compute_count}')
print(f'Radix 计算 token: {radix_engine.compute_count}')
print(f'计算节省: {1 - radix_engine.compute_count/naive_engine.compute_count:.1%}')
print(f'\n基线耗时: {naive_time*1000:.2f} ms')
print(f'Radix 耗时: {radix_time*1000:.2f} ms')
print(f'\nKey: RadixAttention 自动发现任意长度的共享前缀，无需手动指定缓存边界。')
print(f'不同请求组的部分重叠前缀也能被正确去重和复用。')

## 4. 缓存淘汰策略

**为什么需要淘汰？** 显存有限，KV 缓存不能无限增长。当缓存达到上限时，需要选择性地淘汰部分缓存。

**常见策略**：

| 策略 | 原理 | 适用场景 |
|------|------|----------|
| LRU（Least Recently Used） | 淘汰最久未访问的缓存 | 通用，访问局部性强 |
| LFU（Least Frequently Used） | 淘汰访问频率最低的缓存 | 热点前缀明显 |
| Size-aware | 优先淘汰大块缓存 | 显存压力大的场景 |
| Adaptive | 根据访问模式动态切换 | 混合工作负载 |

**RadixAttention 的淘汰**：
- 从叶子节点开始淘汰（不影响其他分支）
- 内部节点在被引用次数归零后才可淘汰
- 淘汰时需更新树结构（合并单链节点）

**显存管理**：
- 设置 KV 缓存总显存上限（如 GPU 显存的 50%）
- 监控缓存命中率，动态调整淘汰策略
- 高命中率 → 保守淘汰；低命中率 → 激进淘汰

In [ ]:
import torch
from collections import OrderedDict
import time

torch.manual_seed(42)


class CacheEntry:
    '''缓存条目。'''
    def __init__(self, key, size):
        self.key = key
        self.size = size
        self.last_access = time.time()
        self.access_count = 0

    def touch(self):
        self.last_access = time.time()
        self.access_count += 1


class CacheEvictionManager:
    '''支持多种淘汰策略的缓存管理器。'''
    def __init__(self, max_size, strategy='lru'):
        self.max_size = max_size
        self.strategy = strategy
        self.entries = OrderedDict()
        self.current_size = 0
        self.evictions = 0
        self.hits = 0
        self.misses = 0

    def _evict(self):
        '''根据策略淘汰缓存直到有足够空间。'''
        while self.current_size > self.max_size and self.entries:
            if self.strategy == 'lru':
                key = min(self.entries, key=lambda k: self.entries[k].last_access)
            elif self.strategy == 'lfu':
                key = min(self.entries, key=lambda k: self.entries[k].access_count)
            elif self.strategy == 'size':
                key = max(self.entries, key=lambda k: self.entries[k].size)
            else:
                key = next(iter(self.entries))
            entry = self.entries.pop(key)
            self.current_size -= entry.size
            self.evictions += 1

    def put(self, key, size):
        '''插入缓存条目。'''
        if key in self.entries:
            self.current_size -= self.entries[key].size
        entry = CacheEntry(key, size)
        self.entries[key] = entry
        self.current_size += size
        if self.current_size > self.max_size:
            self._evict()

    def get(self, key):
        '''访问缓存。'''
        if key in self.entries:
            self.entries[key].touch()
            self.hits += 1
            return True
        self.misses += 1
        return False


def simulate_workload(manager, requests):
    for key, size in requests:
        if not manager.get(key):
            manager.put(key, size)
    return manager


# 模拟缓存压力测试
keys = list(range(20))
sizes = torch.randint(1, 10, (20,)).tolist()

# 访问模式：部分热点 + 长尾
access_pattern = (
    [(0, sizes[0])] * 5 + [(1, sizes[1])] * 4 + [(2, sizes[2])] * 3 +
    [(3, sizes[3])] * 2 + [(k, sizes[k]) for k in range(4, 20)] +
    [(0, sizes[0]), (5, sizes[5]), (10, sizes[10])]
)

max_cache = 40

print('=== 缓存淘汰策略对比 ===')
print(f'工作负载: {len(access_pattern)} 次访问, {len(keys)} 个唯一键')
print(f'总缓存需求: {sum(sizes)} 单位, 缓存上限: {max_cache} 单位')
print()

for strategy in ['lru', 'lfu', 'size']:
    mgr = CacheEvictionManager(max_cache, strategy)
    simulate_workload(mgr, access_pattern)
    total = mgr.hits + mgr.misses
    print(f'[{strategy.upper()}] 命中: {mgr.hits}, 未命中: {mgr.misses}, '
          f'淘汰: {mgr.evictions}, 命中率: {mgr.hits/total:.1%}')

print(f'\nKey: 不同淘汰策略在不同访问模式下表现各异。')
print(f'LRU 适合时间局部性强的负载，LFU 适合热点明显的负载，Size-aware 适合显存紧张场景。')
print(f'实际系统通常采用自适应策略，根据命中率动态调整。')

## 5. 实践：多轮对话优化

**多轮对话是前缀缓存的理想场景**：

每一轮对话的输入 = 系统提示 + 历史所有对话 + 新用户输入

随着对话进行，前缀不断增长，但每一轮的前一轮内容完全相同——天然适合前缀缓存。

**优化效果**：
- 第 1 轮：完整计算（无缓存）
- 第 2 轮：复用第 1 轮的所有内容，仅计算新输入
- 第 N 轮：复用前 N-1 轮的所有内容

**累积加速**：对话越长，缓存复用比例越高，平均每轮计算量越低。

**系统提示缓存**：固定的系统提示在所有对话中共享，可长期缓存（甚至跨会话）。

In [ ]:
import torch
import torch.nn as nn
import math
import time

torch.manual_seed(42)


class ConversationCacheManager:
    '''多轮对话缓存管理器：跨轮次复用前缀。'''
    def __init__(self, d_model=128, n_heads=4):
        self.attn = SimpleAttention(d_model, n_heads)
        self.embed = nn.Embedding(1000, d_model)
        self.cached_prefix_len = 0
        self.cached_tokens = None
        self.compute_count = 0
        self.reused_count = 0

    def _compute(self, token_ids):
        x = self.embed(token_ids.unsqueeze(0))
        start = time.perf_counter()
        _ = self.attn(x)
        return time.perf_counter() - start

    def turn(self, full_context):
        '''处理一轮对话，复用已缓存前缀。'''
        n = full_context.shape[0]
        if self.cached_tokens is None:
            elapsed = self._compute(full_context)
            self.compute_count += n
            self.cached_tokens = full_context.clone()
            self.cached_prefix_len = n
            return elapsed, n, 0

        overlap = min(self.cached_prefix_len, n)
        for i in range(overlap):
            if self.cached_tokens[i].item() != full_context[i].item():
                overlap = i
                break

        if overlap >= n:
            self.reused_count += n
            return 0.0, 0, n

        new_part = full_context[overlap:]
        elapsed = self._compute(new_part)
        self.compute_count += new_part.shape[0]
        self.reused_count += overlap
        self.cached_tokens = full_context.clone()
        self.cached_prefix_len = n
        return elapsed, new_part.shape[0], overlap


# 模拟多轮对话
system_prompt = torch.randint(0, 1000, (64,))
user1 = torch.randint(0, 1000, (20,))
assistant1 = torch.randint(0, 1000, (20,))
user2 = torch.randint(0, 1000, (20,))
assistant2 = torch.randint(0, 1000, (20,))
user3 = torch.randint(0, 1000, (20,))

turns = [
    ('Turn 1', torch.cat([system_prompt, user1])),
    ('Turn 2', torch.cat([system_prompt, user1, assistant1, user2])),
    ('Turn 3', torch.cat([system_prompt, user1, assistant1, user2, assistant2, user3])),
]

print('=== 多轮对话前缀缓存优化 ===')
print(f'系统提示: {system_prompt.shape[0]} tokens')
print()

cached_mgr = ConversationCacheManager()
cached_total_time = 0.0
cached_total_compute = 0
for name, context in turns:
    elapsed, computed, reused = cached_mgr.turn(context)
    cached_total_time += elapsed
    cached_total_compute += computed
    print(f'{name}: 上下文 {context.shape[0]} tokens, '
          f'新计算 {computed}, 复用 {reused}, 耗时 {elapsed*1000:.2f} ms')

naive_total_time = 0.0
naive_total_compute = 0
for name, context in turns:
    elapsed = cached_mgr._compute(context)
    naive_total_time += elapsed
    naive_total_compute += context.shape[0]

print(f'\n--- 汇总 ---')
print(f'基线总计算: {naive_total_compute} tokens, 耗时 {naive_total_time*1000:.2f} ms')
print(f'缓存总计算: {cached_total_compute} tokens, 耗时 {cached_total_time*1000:.2f} ms')
print(f'计算节省: {1 - cached_total_compute/naive_total_compute:.1%}')
print(f'累积复用: {cached_mgr.reused_count} tokens')
print(f'\nKey: 多轮对话中，每轮新增内容远小于累积上下文，前缀缓存收益随对话深入而增大。')
print(f'系统提示在整个会话中被完全复用，是缓存命中率的最大贡献者。')

## 📝 课后思考题

1. 前缀缓存为什么对多轮对话场景特别有效？如果对话中用户频繁切换话题，缓存效果会如何变化？

2. RadixAttention 相比简单的哈希前缀缓存有哪些优势？基数树结构如何自动发现部分重叠的前缀？

3. 当 GPU 显存不足以容纳所有 KV 缓存时，LRU 和 LFU 淘汰策略各有什么优缺点？在什么场景下应该选择哪种？

4. 如果模型在服务过程中被微调更新，前缀缓存需要如何处理？为什么不能直接复用旧缓存？